# Taller de retorno a ResearchOS

**Objetivo.** Recuperar y afianzar lo que ya construiste en V1 antes de arrancar V2. No es evaluación — es reactivación. Si algo no lo sabes, lo buscas en tu propio código en `src/researchos/` y anotas la duda para discutirla después.

**Duración estimada.** 1.5 a 2 horas si no lo apuras. Si vas más rápido, probablemente estás autocompletando; frena.

**Reglas.**

1. No uses Claude Code, Copilot ni ningún autocompletado LLM para las celdas de código. El punto es forzar el recall, no producir código correcto.
2. Sí puedes leer tu propio código en `src/researchos/` cuando te atores — esa búsqueda es parte del ejercicio.
3. Cada sección tiene una celda de "check" al final. Corre el check antes de pasar a la siguiente sección.
4. Las respuestas conceptuales van en celdas markdown propias. Escribe con tus palabras — no copies de docs.
5. Al final, en la sección de auto-verificación, hay soluciones sugeridas. No las mires antes de terminar cada sección.

**Cobertura.** Ocho secciones:

1. Clean Architecture — conceptos base
2. Domain — modelos y Protocols
3. Infrastructure — LLM provider
4. Retrieval — VectorStore y BM25
5. Application — RAG y Hybrid Search
6. Async y concurrencia
7. Testing con mocks
8. Meta arquitectural — extender el sistema

Cuando termines, contame en la sesión qué te costó, qué dudas quedaron y con qué sección te sentiste más flojo. Con eso ajustamos el arranque de V2.

---


## 1. Clean Architecture — conceptos base

Preguntas para responder con tus palabras en la celda markdown que sigue. Objetivo: 5-10 min.


**1.1** ¿Cuál es la regla de dependencia entre las tres capas (`domain`, `application`, `infrastructure`)? Es decir, ¿quién puede importar de quién y quién NO puede importar de quién? ¿Por qué esta regla y qué se rompe si no se respeta?

*Tu respuesta:*

application importa de infrastructure, y infrastructure importa de domain, application no puede importar directamente de domain. Esta regla está para mantener separadas las capas y permitir que la capa infrastructure implemente diferentes estructuras y servicios (por ejemplo, conexiones a diferentes tipos de proveedores de LLM o diferentes BD vectoriales) para que las aplicaciones puedan ser cambiadas fácilmente entre esos proveedores, así los refactores son poco costosos y se puede tener un escalamiento fácil. Si se rompe justamente se pierde la facilidad de escalar y de expandir a distintos proveedores porque el código de infraestructura viviría en la aplicación y tocaría tocar mucho código para hacer cambios; además domain establece los contratos de manera que si cualquier implemnetación en infrastructure cumple el contrato debería ser suficiente para implementarse en una application

---

**1.2** ¿Qué es un `Protocol` de Python y en qué se diferencia de una clase abstracta (`ABC`)? Da un ejemplo concreto de tu proyecto (nombre del Protocol y qué implementaciones concretas lo satisfacen).

*Tu respuesta:*

Un Protocol es un un contrato que a diferencia de una clase abstracta no implementa los métodos o propiedades, sino que declara qué métodos y sus firmas debe tener una implementación. Por ejemplo el Protocolo "LLMProvider" establece que cualquier implementación debe tener los métodos "generate" y "stream" sin importar si la conexión que tenemos es con claude api o gemini o gpt. La implementación en la ruta src/researcos/infrastructure/llm/anthropic_llm.py satisface éste protocolo

---

**1.3** En tu proyecto tienes `Retriever` y `VectorStore` como Protocols separados. `VectorStore` tiene `search` y `upsert`; `Retriever` solo tiene `search`. ¿Por qué esta separación? ¿Qué se te complicaría si `BM25Retriever` implementara `VectorStore` directamente?

*Tu respuesta:*

<!-- escribe acá -->

---

**1.4** ¿Por qué los agentes en `application/agents/` usan composición (importar funciones de `agent_utils.py`) en lugar de herencia (heredar de una clase `BaseAgent`)? Da al menos dos razones concretas.

*Tu respuesta:*

<!-- escribe acá -->


## 2. Domain — modelos y Protocols

Objetivo: 10-15 min. Recuerdas cómo definir modelos Pydantic y Protocols mínimos.


**2.1** Define un modelo Pydantic llamado `Feedback` que representa un feedback de usuario sobre una respuesta del sistema. Debe tener:

- `session_id: str`
- `query: str`  
- `answer: str`
- `rating: int` entre 1 y 5 (usa validación de Pydantic)
- `comment: str` opcional, default vacío
- `created_at: datetime` con default de la fecha actual

Escribe el modelo. Debe importar solo de la biblioteca estándar y de Pydantic.


In [ ]:
from datetime import datetime
from pydantic import BaseModel, Field

# Tu código acá




**2.2** Define un `Protocol` llamado `FeedbackStore` que abstrae el almacenamiento de feedback. Debe tener dos métodos async:

- `save(feedback: Feedback) -> None`
- `list_by_session(session_id: str) -> list[Feedback]`

Escribe el Protocol siguiendo el mismo estilo que ves en `src/researchos/domain/interfaces.py`.


In [ ]:
from typing import Protocol

# Tu código acá




**2.3 — Check.** Corre la celda siguiente. Debe pasar sin error.


In [ ]:
# Check sección 2
try:
    fb = Feedback(session_id="s1", query="q", answer="a", rating=3)
    assert fb.comment == ""
    assert isinstance(fb.created_at, datetime)
    print("2.1 ✓")
except Exception as e:
    print(f"2.1 ✗ {e}")

try:
    # Validación de rating fuera de rango debe fallar
    Feedback(session_id="s1", query="q", answer="a", rating=10)
    print("2.1 rating validation ✗ (aceptó 10)")
except Exception:
    print("2.1 rating validation ✓")

try:
    class DummyStore:
        async def save(self, feedback): pass
        async def list_by_session(self, session_id): return []
    # Structural subtyping: no hay assert directo en runtime, pero debe compilar
    store: FeedbackStore = DummyStore()
    print("2.2 ✓ (structural subtyping se validaría con mypy)")
except NameError as e:
    print(f"2.2 ✗ {e}")


## 3. Infrastructure — LLM Provider

Objetivo: 10-15 min. Recuerdas cómo se conecta un LLM externo a través del Protocol.

Tu proyecto tiene `AnthropicLLM` en `src/researchos/infrastructure/llm/anthropic_llm.py`. Vas a implementar una versión simplificada del método `generate`.


**3.1** Implementa la clase `SimpleAnthropicLLM` que satisface el Protocol `LLMProvider`. El método `generate` debe:

1. Convertir la lista de `Message` domain al formato que espera el SDK de Anthropic (recuerda: el rol `system` se maneja aparte, no va en `messages`).
2. Llamar a `client.messages.create(...)` con el modelo `claude-haiku-4-5`, `max_tokens=1024`.
3. Devolver el texto del primer bloque de la respuesta.

No necesitas implementar `stream` — deja un `raise NotImplementedError`. Tampoco necesitas correr esto contra la API real; solo la estructura.

Pista: mira tu propio código en `infrastructure/llm/anthropic_llm.py` si te atoras. El punto no es memorizar el SDK, es entender el patrón.


In [ ]:
from anthropic import Anthropic
# from researchos.domain.interfaces import LLMProvider  # descomenta si vas a correr esto
# from researchos.domain.models import Message

# Para el taller, redefinimos Message localmente si no lo tienes importado:
from pydantic import BaseModel

class Message(BaseModel):
    role: str
    content: str


class SimpleAnthropicLLM:
    def __init__(self, api_key: str | None = None, model: str = "claude-haiku-4-5"):
        self.client = Anthropic(api_key=api_key)
        self.model = model

    async def generate(self, messages: list[Message]) -> str:
        # 1. Separar el mensaje system (si existe) del resto
        # 2. Construir la lista de messages para el SDK (solo user/assistant)
        # 3. Llamar a self.client.messages.create(...)
        # 4. Devolver el texto del primer bloque
        
        # Tu código acá
        pass

    async def stream(self, messages: list[Message]):
        raise NotImplementedError


**3.2 — Conceptual.** En el Protocol `LLMProvider`, `generate` es `async def`. Pero el SDK de Anthropic (`self.client.messages.create`) es sincrónico. Explica en tus palabras:

(a) ¿Se rompe algo por declarar `async def generate` aunque adentro llames a un método sincrónico?

(b) ¿Cuándo sí importaría convertir la llamada interna a async (con `httpx.AsyncClient` o `anthropic.AsyncAnthropic`)?

*Tu respuesta:*

<!-- escribe acá -->


## 4. Retrieval — VectorStore y BM25

Objetivo: 15 min. Recuerdas cómo funciona la búsqueda vectorial y BM25, y por qué complementan.


**4.1 — Conceptual.** Da un ejemplo concreto de una query donde BM25 supera a la búsqueda vectorial, y otro ejemplo donde vectorial supera a BM25. Los ejemplos deben ser del dominio de ResearchOS (papers de ML/AI). Un párrafo por caso.

*Tu respuesta:*

<!-- escribe acá -->


**4.2** Implementa una clase `SimpleBM25` que satisface el Protocol `Retriever` (solo `search`). Debe:

1. Recibir en `__init__` una `list[Document]` con `doc_id`, `text` y `metadata`.
2. Construir un índice BM25 en memoria (usa `rank_bm25.BM25Okapi`).
3. Tokenizar simple: `text.lower().split()`.
4. En `search(query, k)`, devolver los top-k Documents ordenados por score BM25 descendente, con el campo `score` populado sin mutar los originales.

Para el ejercicio, usa un `Document` local simple (redefinido abajo).


In [ ]:
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field


class Document(BaseModel):
    doc_id: str
    text: str
    metadata: dict = Field(default_factory=dict)
    score: float = 0.0


class SimpleBM25:
    def __init__(self, documents: list[Document]) -> None:
        # Tu código acá: guardar documents, tokenizar corpus, construir BM25Okapi
        pass

    async def search(self, query: str, k: int) -> list[Document]:
        # Tu código acá:
        # 1. Tokenizar la query
        # 2. Obtener scores con self.bm25.get_scores(...)
        # 3. Ordenar índices por score descendente, tomar top-k
        # 4. Devolver los documents correspondientes con score actualizado (SIN MUTAR)
        pass


**4.3 — Check.** Corre la celda. Debe pasar sin error.


In [ ]:
import asyncio

async def _check_bm25():
    docs = [
        Document(doc_id="d1", text="reinforcement learning from human feedback"),
        Document(doc_id="d2", text="convolutional neural networks for images"),
        Document(doc_id="d3", text="human feedback in language model training"),
    ]
    bm25 = SimpleBM25(docs)
    results = await bm25.search("human feedback", k=2)
    
    assert len(results) == 2, f"Expected 2, got {len(results)}"
    assert results[0].doc_id in {"d1", "d3"}, f"Top result should mention 'human feedback', got {results[0].doc_id}"
    assert results[0].score > 0, "Score should be populated"
    # No mutación: doc original de la lista sigue en score=0.0
    assert docs[0].score == 0.0, "Original docs must NOT be mutated"
    print("4.2 ✓")

asyncio.run(_check_bm25())


## 5. Application — RAG y Hybrid Search

Objetivo: 15-20 min. Recuerdas el patrón RAG y el algoritmo RRF que implementaste en `retrieval_service.py`.


**5.1 — Conceptual.** Explica en tus palabras qué es Reciprocal Rank Fusion (RRF) y responde:

(a) ¿Por qué RRF suma las contribuciones cuando un documento aparece en dos rankings, en lugar de promediarlas o quedarse con la mayor?

(b) ¿Por qué el índice del rank arranca en 1 (no en 0)?

(c) ¿Qué controla la constante `rrf_k` (típicamente 60)? Si la subo a 200, ¿qué cambia en la práctica?

*Tu respuesta:*

<!-- escribe acá -->


**5.2** Implementa `hybrid_search_simple` sin mirar tu propio código. Debe:

1. Recibir `query: str`, `retrievers: list[Retriever]`, `k: int = 5`, `rrf_k: int = 60`.
2. Validar que `retrievers` no esté vacío (raise `ValueError`).
3. Ejecutar los retrievers en paralelo con `asyncio.gather`, cada uno pidiendo `k * 2` candidatos.
4. Acumular scores RRF por `doc_id` (`1 / (rrf_k + rank)`), con rank 1-indexed.
5. Devolver los top-k Documents ordenados por RRF descendente, con `score` = RRF score.

Cada documento único aparece una vez en el resultado.


In [ ]:
import asyncio
from typing import Protocol


class Retriever(Protocol):
    async def search(self, query: str, k: int) -> list[Document]:
        ...


async def hybrid_search_simple(
    query: str,
    retrievers: list[Retriever],
    k: int = 5,
    rrf_k: int = 60,
) -> list[Document]:
    # Tu código acá
    pass


**5.3 — Check.** Corre la celda. El documento que aparece en ambos retrievers debe quedar primero.


In [ ]:
class _MockRetriever:
    def __init__(self, docs): self.docs = docs
    async def search(self, query, k): return self.docs[:k]


async def _check_hybrid():
    r_a = _MockRetriever([
        Document(doc_id="doc1", text="t1"),
        Document(doc_id="doc2", text="t2"),
        Document(doc_id="doc3", text="t3"),
    ])
    r_b = _MockRetriever([
        Document(doc_id="doc2", text="t2"),
        Document(doc_id="doc4", text="t4"),
        Document(doc_id="doc5", text="t5"),
    ])
    
    results = await hybrid_search_simple("q", retrievers=[r_a, r_b], k=5)
    
    assert results[0].doc_id == "doc2", f"doc2 should win, got {results[0].doc_id}"
    assert results[0].score > results[1].score, "Top should have higher RRF"
    ids = [r.doc_id for r in results]
    assert len(ids) == len(set(ids)), "Docs must be deduplicated"
    print("5.2 ✓ orden correcto, doc2 primero, sin duplicados")
    
    try:
        await hybrid_search_simple("q", retrievers=[], k=5)
        print("5.2 ✗ retrievers vacío no lanza error")
    except ValueError:
        print("5.2 ✓ retrievers vacío lanza ValueError")


asyncio.run(_check_hybrid())


## 6. Async y concurrencia

Objetivo: 10-15 min. Recuerdas cuándo async ayuda y cuándo no, y los antipatrones comunes.


**6.1 — Conceptual.** Regla mental que documentaste en learnings el 15/04: "¿Esperas algo externo (API, disco, red)? → `async def`. ¿Solo calculas en memoria? → `def` normal". Aplica esa regla a cada función siguiente y justifica:

(a) `overlap_chunking(text, paper_id, chunk_size, overlap) -> list[Chunk]`

(b) `download_pdf(url) -> bytes`

(c) `embed_text(text) -> list[float]` (usa un modelo local en CPU con sentence-transformers)

(d) `send_telegram_message(bot, chat_id, text) -> None`

*Tu respuesta:*

<!-- escribe acá -->


**6.2** El código siguiente pretende descargar tres URLs en paralelo. Tiene un bug conceptual que hace que corra en secuencia. Identifícalo y corrígelo.


In [ ]:
import asyncio
import httpx


async def fetch(client: httpx.AsyncClient, url: str) -> str:
    response = await client.get(url)
    return response.text


async def fetch_all_broken(urls: list[str]) -> list[str]:
    # Este código está mal — corre secuencial. Explica por qué y arregla.
    async with httpx.AsyncClient() as client:
        results = []
        for url in urls:
            result = await fetch(client, url)
            results.append(result)
        return results


# Escribe la versión corregida acá
async def fetch_all_fixed(urls: list[str]) -> list[str]:
    # Tu código acá
    pass


# Explicación del bug (celda markdown abajo):


*Tu explicación del bug:*

<!-- escribe acá qué estaba mal y por qué tu versión sí paraleliza -->


## 7. Testing con mocks

Objetivo: 10-15 min. Recuerdas cómo se testea el application layer sin tocar sistemas reales.


**7.1 — Conceptual.** ¿Cuál es la diferencia entre `@pytest.mark.unit` y `@pytest.mark.integration` en tu proyecto? Da un ejemplo concreto de cada uno y explica qué haría CI si separaras el pipeline en dos etapas.

*Tu respuesta:*

<!-- escribe acá -->


**7.2** Escribe un `MockFeedbackStore` en memoria que satisfaga el Protocol `FeedbackStore` que definiste en la sección 2. Debe funcionar como el `MockVectorStore` de tu `conftest.py` — sin BD real, guardando en un dict interno.


In [ ]:
# Recordatorio de tu Feedback y FeedbackStore de la sección 2
# (los redefiníamos abajo si te acomoda tenerlos a la mano)


class MockFeedbackStore:
    def __init__(self):
        # Tu código acá: estructura interna para almacenar por session
        pass

    async def save(self, feedback) -> None:
        # Tu código acá
        pass

    async def list_by_session(self, session_id: str) -> list:
        # Tu código acá
        pass


**7.3** Escribe un test `test_mock_feedback_store_saves_and_lists` que:

1. Crea un `MockFeedbackStore`
2. Guarda dos `Feedback` de la misma sesión y uno de otra sesión
3. Verifica que `list_by_session` de la primera sesión devuelva solo dos, en el orden en que se guardaron

No necesitas `@pytest.mark.asyncio` acá — corre con `asyncio.run` como los otros checks del taller.


In [ ]:
async def test_mock_feedback_store_saves_and_lists():
    # Tu código acá
    pass


asyncio.run(test_mock_feedback_store_saves_and_lists())
print("7.3 ✓ si llegaste hasta acá sin AssertionError")


## 8. Meta arquitectural — extender el sistema

Objetivo: 15 min. Estas son las preguntas que definen si internalizaste la arquitectura o solo la seguiste. No hay código — solo diseño en palabras.


**8.1** Alguien te pide agregar Qdrant como vector store alternativo a Chroma. En tu proyecto tal cual está hoy, ¿qué archivos habría que crear o modificar? Sé concreto: rutas y qué va en cada uno. ¿Qué archivos NO deberías tocar?

*Tu respuesta:*

<!-- escribe acá -->

---

**8.2** Quieres agregar un canal nuevo: Slack. ¿Dónde va el código de Slack? ¿Qué relación tiene con `application/services/rag_service.py`? Traza el flujo de un mensaje entrando por Slack hasta la respuesta.

*Tu respuesta:*

<!-- escribe acá -->

---

**8.3** El PO de Pensiones (Paola) te pide un endpoint HTTP que reciba un caso y devuelva una recomendación. En términos de tu arquitectura, ¿en qué capa vive un endpoint HTTP? ¿Cuál es el rol de FastAPI: parte del motor o parte del canal?

*Tu respuesta:*

<!-- escribe acá -->

---

**8.4 — La pregunta clave.** En una entrevista de AI Engineering te preguntan: "Explica cómo evaluarías un sistema RAG en producción, sin data leakage". Con lo que sabes hoy, escribe una respuesta de 3-5 oraciones. Menciona: qué mides, cómo obtienes ground truth, qué haces con las queries que fallan.

*Tu respuesta:*

<!-- escribe acá -->


---

## Apéndice: auto-verificación

No mires esta sección antes de terminar. Es orientativa — hay más de una forma correcta.

### 1. Clean Architecture

**1.1.** Dependencia: `infrastructure → application → domain`, nunca al revés. `domain` no importa nada de las otras dos; `application` importa de `domain` (Protocols y modelos) pero no de `infrastructure`; `infrastructure` importa de `domain` para implementar los Protocols. Si `domain` importa de `infrastructure`, cambiar de Chroma a Qdrant te obliga a tocar código de negocio, y los tests unitarios de dominio dejan de correrse sin instalar todo el stack.

**1.2.** Un Protocol define un contrato estructural. Cualquier clase con los métodos de la firma correcta lo satisface, sin heredar. ABC exige herencia explícita (`class Foo(ABC):`). Ejemplo: `VectorStore` con `ChromaVectorStore` implementándolo por duck typing.

**1.3.** `BM25Retriever` no puede persistir (no tiene `upsert` real, es en memoria). Si implementara `VectorStore`, mentiría sobre su contrato. Separar `Retriever` (solo `search`) permite que `BM25Retriever` sea honesto sobre lo que hace, y que `hybrid_search` reciba una lista genérica de retrievers sin distinguir tipos.

**1.4.** (a) Cada agente es autoresponsable — no hay comportamiento oculto heredado; (b) testear una función pura es más simple que testear un método con `super().__init__` de por medio; (c) agregar un método a `agent_utils.py` no obliga a todos los agentes a adoptarlo.

### 2. Domain

```python
class Feedback(BaseModel):
    session_id: str
    query: str
    answer: str
    rating: int = Field(ge=1, le=5)
    comment: str = ""
    created_at: datetime = Field(default_factory=datetime.now)


class FeedbackStore(Protocol):
    async def save(self, feedback: Feedback) -> None: ...
    async def list_by_session(self, session_id: str) -> list[Feedback]: ...
```

### 3. LLM

Estructura mínima del `generate`:

```python
system_msg = next((m.content for m in messages if m.role == "system"), None)
user_msgs = [{"role": m.role, "content": m.content} for m in messages if m.role != "system"]
kwargs = {"model": self.model, "max_tokens": 1024, "messages": user_msgs}
if system_msg: kwargs["system"] = system_msg
response = self.client.messages.create(**kwargs)
return response.content[0].text
```

**3.2.** (a) No se rompe: `async def` con contenido sincrónico se resuelve inmediatamente. Es válido. (b) Importa cuando llamas al LLM en paralelo con otras coroutines (p.ej. varios agentes en `asyncio.gather`), porque un `create` sincrónico bloquea el event loop y anula el paralelismo.

### 4. Retrieval

**4.1.** BM25 supera a vectorial en queries con siglas o nombres propios: `"BERT vs GPT-3"` (coincidencia exacta). Vectorial supera a BM25 en queries semánticas: `"papers on models that reason step by step"` (encuentra chain-of-thought aunque no diga esas palabras).

Estructura de `SimpleBM25.search`:

```python
tokens = query.lower().split()
scores = self.bm25.get_scores(tokens)
top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
return [self.documents[i].model_copy(update={"score": float(scores[i])}) for i in top_idx]
```

### 5. Hybrid

**5.1.** (a) La suma premia consenso: docs que dos retrievers coinciden en rankear alto suben más que docs que solo uno rankeó alto. Promediar diluiría la señal; quedarse con la mayor ignoraría el consenso. (b) Con rank 0-indexed, el primer doc daría `1/rrf_k` (relativamente grande) y los demás caerían muy rápido. 1-indexed suaviza la curva y es la convención del paper original. (c) `rrf_k` controla la suavidad. Con `rrf_k=60`, el doc en rank 1 aporta `1/61 ≈ 0.0164`; en rank 10, `1/70 ≈ 0.0143`. Con `rrf_k=200`, la diferencia entre rank 1 y rank 10 se aplasta — todos aportan casi lo mismo. Un `rrf_k` alto hace que la fusión sea más democrática entre retrievers; uno bajo premia más al top.

### 6. Async

**6.1.** (a) `def` normal, memoria pura. (b) `async def`, red. (c) `def` normal, CPU en memoria (o async si delegas a un servidor de embeddings). (d) `async def`, red.

**6.2.** El bug es que `await` dentro de un `for` secuencia las llamadas. La corrección:

```python
async def fetch_all_fixed(urls):
    async with httpx.AsyncClient() as client:
        return await asyncio.gather(*[fetch(client, u) for u in urls])
```

### 7. Testing

**7.1.** `unit` no toca red, disco ni servicios externos — corre en milisegundos. `integration` sí — más lento, se corre menos frecuente. En CI, `make test` corre unit en cada commit; `make test-all` corre integración solo en merge a main o nightly.

**7.2.**

```python
class MockFeedbackStore:
    def __init__(self):
        self._by_session: dict[str, list[Feedback]] = {}

    async def save(self, feedback):
        self._by_session.setdefault(feedback.session_id, []).append(feedback)

    async def list_by_session(self, session_id):
        return list(self._by_session.get(session_id, []))
```

### 8. Meta

**8.1.** Crear: `infrastructure/retrieval/qdrant.py` con clase `QdrantVectorStore` que satisface `VectorStore`. Modificar: `config.py` (agregar opción `Literal["chroma", "qdrant"]` en settings) y factory/DI donde se instancia el vector store. NO tocar: `domain/`, `application/services/*` (ni siquiera `retrieval_service.py`), tests de aplicación (los mocks siguen sirviendo).

**8.2.** Slack va en `infrastructure/bot/slack.py`. Recibe mensaje → llama a `rag_service.answer_query(text, llm, store)` → devuelve el string → lo envía a Slack. `rag_service` no sabe que existe Slack, es agnóstico al canal.

**8.3.** El endpoint HTTP vive en `infrastructure/api/routers/`. FastAPI es canal, no motor — expone el mismo motor (`rag_service`, agentes) por HTTP. La lógica de recomendación no sabe si vino de FastAPI, Telegram o un CLI.

**8.4.** Referencia (una versión posible): "Medir faithfulness (¿el LLM inventa cosas no soportadas por los docs?), context precision (¿los docs recuperados son útiles?) y answer relevancy. Ground truth: LLM-as-judge con un modelo distinto al que genera (evita sesgo de auto-evaluación); complementado con eval humano sobre una muestra. Las queries que fallan van a un dataset de regresión: se etiquetan manualmente, se agregan al eval automático, y el pipeline CI/CD bloquea deploys que hagan bajar el score. Diferenciar métricas offline (dataset fijo, corren en cada PR) de online (feedback en producción, thumbs up/down, latencia real)."

---

## Cierre

Cuando termines, en el chat contame:

- Qué secciones te salieron sin friction.
- Qué secciones te forzaron a abrir tu propio código en `src/researchos/`.
- Qué preguntas quedaron sin respuesta clara (esas son los huecos reales).
- Qué error de tus respuestas te sorprendió al comparar con el apéndice.

Con eso ajustamos el arranque de V2 y decidimos si conviene afianzar algún tema antes de tocar código nuevo.
